In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/inference'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Posterior Age-Size Soay Sheep IPMs

Construct age-size GLM and GP IPMs from posterior draws, calculate population-level metrics, compare them with the generating IPM, and save the results. Ages 0 to 7 are explicit and age 8 is the greybeard class.

In [ ]:
from pathlib import Path
import gc
import sys

project_dir = PROJECT_ROOT
sys.path.insert(0, str(project_dir))

import gpflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs

from experiments.soay.common.ipm_utils import make_gp_ipm_agesize, mk_K_glm_agesize, reproduction_fn_glm

In [ ]:
L, U, n_mesh = 2.0, 3.5, 250
max_age = 7
min_reproductive_age = 1
female_prob = 0.5
n_draws = 5000
random_seed = 123
force_recompute = False

n_ages = max_age + 2
n_states = n_ages * n_mesh
ages = np.arange(n_ages)
mesh = L + (np.arange(n_mesh) + 0.5) * (U - L) / n_mesh

data_path = GENERATED / 'soay/age_size/data/gp_ibm_dataset_agesize.csv'
true_metrics_path = GENERATED / 'soay/age_size/data/true_gp_metrics.npz'
glm_samples_path = Path.cwd() / "gpdf_agesize_glm_samples.npz"
gp_samples_path = Path.cwd() / "gpdf_agesize_gp_samples.npz"
output_path = Path.cwd() / "posterior_ipm_metrics_agesize.npz"

data = pd.read_csv(data_path)
with np.load(true_metrics_path) as saved:
    true_metrics = {name: saved[name] for name in saved.files}

with np.load(glm_samples_path) as saved:
    glm_draw_count = min(saved[name].shape[0] for name in saved.files)
with np.load(gp_samples_path) as saved:
    gp_draw_count = min(saved[name].shape[0] for name in saved.files)

available_draws = min(glm_draw_count, gp_draw_count)
if n_draws > available_draws:
    raise ValueError(f"Requested {n_draws} draws, but only {available_draws} are available")

rng = np.random.default_rng(random_seed)
posterior_indices = np.sort(rng.choice(available_draws, size=n_draws, replace=False))

with np.load(glm_samples_path) as saved:
    glm_samples = {name: saved[name][posterior_indices] for name in saved.files}
with np.load(gp_samples_path) as saved:
    gp_samples = {name: saved[name][posterior_indices] for name in saved.files if not name.endswith("_latent")}

if np.asarray(true_metrics["omega"]).size != n_states:
    raise ValueError(
        f"True metrics contain {np.asarray(true_metrics['omega']).size} states; expected {n_states}"
    )

print(f"Age Classes: {n_ages} ({ages[0]}-{ages[-1]}, With Age {ages[-1]} As Greybeard)")
print(f"IPM Dimensions: {n_states} x {n_states}")
print(f"Posterior Draws: {n_draws} Of {available_draws}")

In [ ]:
abc_results_path = GENERATED / 'soay/age_size/abc/gp/abc_samples.npz'
deeprv_results_path = GENERATED / 'soay/age_size/abc/deeprv/abc_samples.npz'
rate_columns = {"growth": 0, "surv": 1, "repr": 2, "recr": 3, "rcsz": 4}

def load_abc_samples(results_path, seed):
    with np.load(results_path) as saved:
        particles = saved["indices"]
        weights = saved["weights"].astype(float)
    weights /= weights.sum()
    if particles.shape[1] != len(rate_columns):
        raise ValueError(f"Expected five vital-rate indices, received {particles.shape[1]}")
    rng = np.random.default_rng(seed)
    particle_rows = rng.choice(len(particles), size=n_draws, replace=True, p=weights)
    posterior_indices = particles[particle_rows]
    selected_samples = {}
    with np.load(gp_samples_path) as saved:
        for rate, column in rate_columns.items():
            rate_indices = posterior_indices[:, column]
            for name in saved.files:
                if name.startswith(f"{rate}_") and not name.endswith("_latent"):
                    selected_samples[name] = saved[name][rate_indices]
    return particles, weights, particle_rows, posterior_indices, selected_samples

abc_particles, abc_weights, abc_particle_rows, abc_posterior_indices, abc_gp_samples = load_abc_samples(abc_results_path, random_seed)
deeprv_particles, deeprv_weights, deeprv_particle_rows, deeprv_posterior_indices, deeprv_gp_samples = load_abc_samples(deeprv_results_path, random_seed + 1)

print(f"ABC GP Particles: {len(abc_particles)}")
print(f"ABC DeepRV Particles: {len(deeprv_particles)}")
print(f"ABC GP Unique Particles Drawn: {len(np.unique(abc_particle_rows))}")
print(f"ABC DeepRV Unique Particles Drawn: {len(np.unique(deeprv_particle_rows))}")

In [ ]:
plt.style.use("seaborn-v0_8-white")

posterior_samples = {
    "ABC GP": abc_gp_samples,
    "ABC DeepRV": deeprv_gp_samples,
}

colors = {
    "ABC GP": "#4C78A8",
    "ABC DeepRV": "#E07B39",
}

rates = ["surv", "repr", "growth", "recr", "rcsz"]
rate_labels = {
    "surv": "Survival",
    "repr": "Reproduction",
    "growth": "Growth",
    "recr": "Recruitment",
    "rcsz": "Recruit Size",
}

parameters = [
    ("Intercept", "int", {rate: None for rate in rates}),
    ("Variance", "var", {rate: None for rate in rates}),
    ("Size Lengthscale", "ls", {"surv": 0, "repr": 0, "growth": 0, "rcsz": None}),
    ("Age Lengthscale", "ls", {"surv": 1, "repr": 1, "growth": 1, "recr": None}),
]

fig, axes = plt.subplots(
    len(parameters),
    len(rates),
    figsize=(14, 9.5),
    constrained_layout=True,
)

for row, (parameter_label, suffix, dimensions) in enumerate(parameters):
    for column, rate in enumerate(rates):
        axis = axes[row, column]

        if rate not in dimensions:
            axis.axis("off")
            continue

        values = {}
        for method, samples in posterior_samples.items():
            parameter_values = np.asarray(samples[f"{rate}_{suffix}"])
            dimension = dimensions[rate]
            if dimension is not None:
                parameter_values = parameter_values[:, dimension]
            values[method] = parameter_values.reshape(-1)

        combined = np.concatenate(list(values.values()))
        bins = np.histogram_bin_edges(combined, bins=35)

        for method in posterior_samples:
            axis.hist(
                values[method],
                bins=bins,
                density=True,
                color=colors[method],
                linewidth=1.5,
                alpha=0.9,
                label=method,
            )

        if row == 0:
            axis.set_title(rate_labels[rate], fontsize=11)

        if column == 0:
            axis.set_ylabel(f"{parameter_label}\nDensity", fontsize=10)
        else:
            axis.set_ylabel("")

        axis.tick_params(labelsize=8, direction="out")
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.03),
    ncol=2,
    frameon=False,
    fontsize=10,
)

fig.suptitle("ABC Hyperparameter Posterior Distributions", fontsize=14)

#fig.savefig("abc_hyperparameter_posteriors_agesize.pdf", bbox_inches="tight")
#fig.savefig("abc_hyperparameter_posteriors_agesize.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
survival = data[["z", "age", "Surv"]].dropna()
reproduction = data.loc[data["age"] >= min_reproductive_age, ["z", "age", "Repr"]].dropna()
growth = data[["z", "age", "z1"]].dropna()
recruitment = data[["age", "Recr"]].dropna()
recruit_size = data[["z", "Rcsz"]].dropna()

X_surv = survival[["z", "age"]].to_numpy(np.float64)
X_repr = reproduction[["z", "age"]].to_numpy(np.float64)
X_growth = growth[["z", "age"]].to_numpy(np.float64)
X_recr = recruitment[["age"]].to_numpy(np.float64)
X_rcsz = recruit_size[["z"]].to_numpy(np.float64)

training_inputs = {
    "surv": X_surv,
    "repr": X_repr,
    "growth": X_growth,
    "recr": X_recr,
    "rcsz": X_rcsz,
}

for rate, X_train in training_inputs.items():
    expected = gp_samples[f"{rate}_mu"].shape[1]
    if X_train.shape[0] != expected:
        raise ValueError(f"{rate} has {X_train.shape[0]} training rows but {expected} saved GP values")

pd.DataFrame({
    "Observations": {rate: len(X_train) for rate, X_train in training_inputs.items()},
    "Input Dimensions": {rate: X_train.shape[1] for rate, X_train in training_inputs.items()},
})

In [ ]:
def draw(samples, index):
    return {name: value[index] for name, value in samples.items()}


def allocate_metrics():
    return {
        "lambda": np.empty(n_draws),
        "omega": np.empty((n_draws, n_states)),
        "omega_rep": np.empty((n_draws, n_states)),
        "gamma": np.empty((n_draws, n_states)),
    }


def store_metrics(target, index, metrics):
    for name in target:
        target[name][index] = metrics[name]


def calculate_ipm_metrics_sparse(K, reproduction_probability, tolerance=1e-9):
    matrix = csr_matrix(K)
    initial_vector = np.ones(matrix.shape[0])
    eigenvalues, right_vectors = eigs(matrix, k=1, which="LR", v0=initial_vector, tol=tolerance)
    _, left_vectors = eigs(matrix.T, k=1, which="LR", v0=initial_vector, tol=tolerance)

    lambda_ = float(eigenvalues[0].real)
    omega = right_vectors[:, 0].real
    omega *= -1 if omega.sum() < 0 else 1
    omega = np.maximum(omega, 0)
    omega /= omega.sum()

    gamma = left_vectors[:, 0].real
    gamma *= -1 if gamma.sum() < 0 else 1
    if np.isclose(gamma[0], 0):
        raise RuntimeError("The first reproductive-value element is numerically zero")
    gamma /= gamma[0]

    omega_rep = np.asarray(reproduction_probability) * omega
    if omega_rep.sum() <= 0:
        raise RuntimeError("The reproductive stable distribution has zero mass")
    omega_rep /= omega_rep.sum()

    return {"lambda": lambda_, "omega": omega, "omega_rep": omega_rep, "gamma": gamma}


def glm_reproduction_probability(params):
    probability = np.zeros(n_states)
    for age in range(min_reproductive_age, n_ages):
        block = slice(age * n_mesh, (age + 1) * n_mesh)
        probability[block] = reproduction_fn_glm(mesh, params, age=age, use_age=True)
    return probability


def validate_ipm(ipm):
    expected_shape = (n_states, n_states)
    for name in ("K", "P", "F"):
        if ipm[name].shape != expected_shape:
            raise ValueError(f"{name} has shape {ipm[name].shape}; expected {expected_shape}")
        if not np.isfinite(ipm[name]).all():
            raise ValueError(f"{name} contains non-finite values")

    for source_age in range(n_ages):
        source = slice(source_age * n_mesh, (source_age + 1) * n_mesh)
        next_age = source_age + 1 if source_age < n_ages - 1 else source_age
        for target_age in range(n_ages):
            target = slice(target_age * n_mesh, (target_age + 1) * n_mesh)
            if target_age != next_age and np.any(ipm["P"][target, source]):
                raise ValueError(f"Invalid P block from age {source_age} to age {target_age}")
            fecundity_allowed = target_age == 0 and source_age >= min_reproductive_age
            if not fecundity_allowed and np.any(ipm["F"][target, source]):
                raise ValueError(f"Invalid F block from age {source_age} to age {target_age}")


metric_names = ("lambda", "omega", "omega_rep", "gamma")
metric_prefixes = ("glm", "gp", "abc", "deeprv")
required_cache_keys = {f"{prefix}_{name}" for prefix in metric_prefixes for name in metric_names}
required_cache_keys.update({"posterior_indices", "abc_posterior_indices", "deeprv_posterior_indices", "n_mesh", "max_age", "min_reproductive_age", "lower_size", "upper_size"})


def load_metric_group(saved, prefix):
    return {name: np.asarray(saved[f"{prefix}_{name}"]) for name in metric_names}


use_cached_metrics = False
if output_path.exists() and not force_recompute:
    with np.load(output_path) as saved:
        use_cached_metrics = required_cache_keys.issubset(saved.files)
        use_cached_metrics &= int(saved["n_mesh"]) == n_mesh and int(saved["max_age"]) == max_age
        use_cached_metrics &= int(saved["min_reproductive_age"]) == min_reproductive_age
        use_cached_metrics &= np.isclose(float(saved["lower_size"]), L) and np.isclose(float(saved["upper_size"]), U)
        use_cached_metrics &= np.array_equal(saved["posterior_indices"], posterior_indices)
        use_cached_metrics &= np.array_equal(saved["abc_posterior_indices"], abc_posterior_indices)
        use_cached_metrics &= np.array_equal(saved["deeprv_posterior_indices"], deeprv_posterior_indices)
        if use_cached_metrics:
            glm_metrics = load_metric_group(saved, "glm")
            gp_metrics = load_metric_group(saved, "gp")
            abc_metrics = load_metric_group(saved, "abc")
            deeprv_metrics = load_metric_group(saved, "deeprv")

print(f"Metric Cache: {'Loaded' if use_cached_metrics else 'Recomputation Required'}")


In [ ]:
if not use_cached_metrics:
    glm_metrics = allocate_metrics()
    for i in range(n_draws):
        params = draw(glm_samples, i) | {"female_prob": female_prob}
        ipm = mk_K_glm_agesize(n_mesh, params, L, U, max_age, correction=True, use_age=True, min_reproductive_age=min_reproductive_age)
        if i == 0:
            validate_ipm(ipm)
        store_metrics(glm_metrics, i, calculate_ipm_metrics_sparse(ipm["K"], glm_reproduction_probability(params)))
        del ipm
        if (i + 1) % 25 == 0 or i == 0:
            print(f"GLM Draw: {i + 1}/{n_draws}")
            gc.collect()

In [ ]:
class MCMCConditionedGP:
    def __init__(self, X_train, mu_train, variance, lengthscales, intercept, likelihood, jitter=5e-4):
        dtype = gpflow.default_float()
        X_train = np.asarray(X_train, dtype=np.float64)
        if X_train.ndim == 1:
            X_train = X_train.reshape(-1, 1)
        self.input_dim = X_train.shape[1]
        self.X_train = tf.convert_to_tensor(X_train, dtype=dtype)
        self.intercept = tf.convert_to_tensor(intercept, dtype=dtype)
        self.likelihood = likelihood

        lengthscales = np.asarray(lengthscales, dtype=np.float64).reshape(-1)
        if lengthscales.size != self.input_dim:
            raise ValueError(f"Received {lengthscales.size} lengthscales for {self.input_dim} inputs")

        centered = np.asarray(mu_train, dtype=np.float64).reshape(-1, 1) - float(intercept)
        self.f_train_centered = tf.convert_to_tensor(centered, dtype=dtype)
        self.kernel = gpflow.kernels.SquaredExponential(
            variance=float(variance),
            lengthscales=lengthscales,
        )
        n_train = int(self.X_train.shape[0])
        covariance = self.kernel(self.X_train, self.X_train) + tf.cast(jitter, dtype) * tf.eye(n_train, dtype=dtype)
        self.cholesky = tf.linalg.cholesky(covariance)
        self.alpha = tf.linalg.cholesky_solve(self.cholesky, self.f_train_centered)

    def _prepare_inputs(self, X):
        X = tf.convert_to_tensor(X, dtype=gpflow.default_float())
        if X.shape.rank == 0:
            X = tf.reshape(X, (1, 1))
        elif X.shape.rank == 1:
            X = tf.reshape(X, (-1, 1)) if self.input_dim == 1 else tf.reshape(X, (1, -1))
        if X.shape.rank != 2 or X.shape[1] != self.input_dim:
            raise ValueError(f"Expected inputs with {self.input_dim} columns, received shape {X.shape}")
        return X

    def predict_f(self, X, full_cov=False, full_output_cov=False):
        if full_cov or full_output_cov:
            raise NotImplementedError("Only marginal predictive variances are required here")
        X = self._prepare_inputs(X)
        cross_covariance = self.kernel(self.X_train, X)
        mean = tf.linalg.matmul(cross_covariance, self.alpha, transpose_a=True) + self.intercept
        solved = tf.linalg.triangular_solve(self.cholesky, cross_covariance)
        variance = self.kernel.K_diag(X) - tf.reduce_sum(tf.square(solved), axis=0)
        variance = tf.reshape(tf.maximum(variance, tf.cast(0, variance.dtype)), (-1, 1))
        return mean, variance

    def predict_y(self, X, full_cov=False, full_output_cov=False):
        X = self._prepare_inputs(X)
        mean, variance = self.predict_f(X)
        return self.likelihood.predict_mean_and_var(X, mean, variance)


def make_model(rate, X_train, params, likelihood):
    return MCMCConditionedGP(
        X_train,
        params[f"{rate}_mu"],
        params[f"{rate}_var"],
        params[f"{rate}_ls"],
        params[f"{rate}_int"],
        likelihood,
    )


def make_gp_functions(params):
    return {
        "surv": make_model("surv", X_surv, params, gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)),
        "repr": make_model("repr", X_repr, params, gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)),
        "growth": make_model("growth", X_growth, params, gpflow.likelihoods.Gaussian(variance=float(params["growth_noise"]) ** 2)),
        "recr": make_model("recr", X_recr, params, gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)),
        "rcsz": make_model("rcsz", X_rcsz, params, gpflow.likelihoods.Gaussian(variance=float(params["rcsz_noise"]) ** 2)),
    }

In [ ]:
if not use_cached_metrics:
    gp_metrics = allocate_metrics()
    for i in range(n_draws):
        params = draw(gp_samples, i) | {"female_prob": female_prob}
        funcs = make_gp_functions(params)
        ipm = make_gp_ipm_agesize(n_mesh, L, U, max_age, params, funcs, correction=True, min_reproductive_age=min_reproductive_age)
        if i == 0:
            validate_ipm(ipm)
        store_metrics(gp_metrics, i, calculate_ipm_metrics_sparse(ipm["K"], ipm["reproduction_probability"]))
        del funcs, ipm
        if (i + 1) % 25 == 0 or i == 0:
            print(f"GP Draw: {i + 1}/{n_draws}")
            gc.collect()

In [ ]:
def calculate_abc_metrics(samples, label):
    target = allocate_metrics()
    for i in range(n_draws):
        params = draw(samples, i) | {"female_prob": female_prob}
        funcs = make_gp_functions(params)
        ipm = make_gp_ipm_agesize(n_mesh, L, U, max_age, params, funcs, correction=True, min_reproductive_age=min_reproductive_age)
        if i == 0:
            validate_ipm(ipm)
        store_metrics(target, i, calculate_ipm_metrics_sparse(ipm["K"], ipm["reproduction_probability"]))
        del funcs, ipm
        if (i + 1) % 25 == 0 or i == 0:
            print(f"{label} Draw: {i + 1}/{n_draws}")
            gc.collect()
    return target

if not use_cached_metrics:
    abc_metrics = calculate_abc_metrics(abc_gp_samples, "ABC GP")
    deeprv_metrics = calculate_abc_metrics(deeprv_gp_samples, "ABC DeepRV")

In [ ]:
if not use_cached_metrics:
    np.savez_compressed(
        output_path,
        posterior_indices=posterior_indices, mesh_points=mesh, ages=ages, max_age=np.array(max_age),
        min_reproductive_age=np.array(min_reproductive_age), lower_size=np.array(L), upper_size=np.array(U), n_mesh=np.array(n_mesh),
        glm_lambda=glm_metrics["lambda"], glm_omega=glm_metrics["omega"], glm_omega_rep=glm_metrics["omega_rep"], glm_gamma=glm_metrics["gamma"],
        gp_lambda=gp_metrics["lambda"], gp_omega=gp_metrics["omega"], gp_omega_rep=gp_metrics["omega_rep"], gp_gamma=gp_metrics["gamma"],
        abc_particle_rows=abc_particle_rows, abc_posterior_indices=abc_posterior_indices, abc_source_weights=abc_weights,
        abc_lambda=abc_metrics["lambda"], abc_omega=abc_metrics["omega"], abc_omega_rep=abc_metrics["omega_rep"], abc_gamma=abc_metrics["gamma"],
        deeprv_particle_rows=deeprv_particle_rows, deeprv_posterior_indices=deeprv_posterior_indices, deeprv_source_weights=deeprv_weights,
        deeprv_lambda=deeprv_metrics["lambda"], deeprv_omega=deeprv_metrics["omega"], deeprv_omega_rep=deeprv_metrics["omega_rep"], deeprv_gamma=deeprv_metrics["gamma"],
    )
    print(f"Saved: {output_path}")
else:
    print(f"Reused: {output_path}")

In [ ]:
def normalise(values, epsilon=1e-12):
    values = np.clip(np.asarray(values), epsilon, None)
    return values / values.sum(axis=-1, keepdims=True)


def evaluate_ipm_models(model_metrics, true_metrics):
    true_lambda = float(true_metrics["lambda"])
    true_omega = normalise(true_metrics["omega"])
    true_omega_rep = normalise(true_metrics["omega_rep"])
    true_gamma = np.asarray(true_metrics["gamma"])
    results = {}

    for model, metrics in model_metrics.items():
        lambda_values = np.asarray(metrics["lambda"])
        lambda_bias = lambda_values.mean() - true_lambda
        lambda_variance = lambda_values.var()
        lambda_mse = np.mean((lambda_values - true_lambda) ** 2)
        omega = normalise(metrics["omega"])
        omega_rep = normalise(metrics["omega_rep"])
        gamma = np.asarray(metrics["gamma"])
        results[model] = {
            "Lambda Mean": lambda_values.mean(),
            "Lambda SD": lambda_values.std(),
            "Lambda Bias": lambda_bias,
            "Lambda Bias Squared": lambda_bias ** 2,
            "Lambda Variance": lambda_variance,
            "Lambda MSE": lambda_mse,
            "Stable State KL": np.mean(np.sum(true_omega * np.log(true_omega / omega), axis=1)),
            "Reproductive State KL": np.mean(np.sum(true_omega_rep * np.log(true_omega_rep / omega_rep), axis=1)),
            "Reproductive Value Cosine": np.mean(
                np.sum(gamma * true_gamma, axis=1)
                / (np.linalg.norm(gamma, axis=1) * np.linalg.norm(true_gamma))
            ),
        }

    return pd.DataFrame(results).T


model_metrics = {"GLM": glm_metrics, "GP": gp_metrics, 'ABC GP': abc_metrics, "ABC DeepRV": deeprv_metrics}
results_table = evaluate_ipm_models(model_metrics, true_metrics)
results_table

In [ ]:
plt.style.use("seaborn-v0_8-white")
colors = {
    "GLM": "#4C78A8",         # Blue
    "GP": "#E07B39",          # Orange
    "ABC GP": "#59A14F",      # Green
    "ABC DeepRV": "#B07AA1",  # Purple
}
all_lambdas = np.concatenate([metrics["lambda"] for metrics in model_metrics.values()])
bins = np.linspace(all_lambdas.min(), all_lambdas.max(), 50)

fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
for model, metrics in model_metrics.items():
    axis.hist(metrics["lambda"], bins=bins, density=True, alpha=0.7, linewidth=1.8, color=colors[model], label=model)
axis.axvline(float(true_metrics["lambda"]), color="black", linestyle="--", linewidth=1.5, label="True")
axis.set_title("Posterior Population Growth Rate")
axis.set_xlabel("Population Growth Rate")
axis.set_ylabel("Density")
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(False)
axis.legend(frameon=False)
plt.show()

In [ ]:
import seaborn as sns

true_omega = normalise(true_metrics["omega"])
true_omega_rep = normalise(true_metrics["omega_rep"])
true_gamma = np.asarray(true_metrics["gamma"])
sample_scores = {}

for model, metrics in model_metrics.items():
    omega = normalise(metrics["omega"])
    omega_rep = normalise(metrics["omega_rep"])
    gamma = np.asarray(metrics["gamma"])
    sample_scores[model] = {
        "Stable-State KL Distance": np.sum(true_omega * np.log(true_omega / omega), axis=1),
        "Reproductive-State KL Distance": np.sum(true_omega_rep * np.log(true_omega_rep / omega_rep), axis=1),
        "Reproductive-Value Cosine Similarity": np.sum(gamma * true_gamma, axis=1) / (np.linalg.norm(gamma, axis=1) * np.linalg.norm(true_gamma)),
    }

plt.style.use("seaborn-v0_8-white")
colors = {"GLM": "#0072B2", "GP": "#D55E00", "ABC GP": "#009E73", "ABC DeepRV": "#CC79A7"}
score_names = list(sample_scores["GLM"])
fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.5), constrained_layout=True)

for axis, score_name in zip(axes, score_names):
    for model in model_metrics:
        bounds = (-1, 1) if "Cosine" in score_name else (0, None)
        sns.kdeplot(sample_scores[model][score_name], ax=axis, color=colors[model], linewidth=1.4, fill=True, alpha=0.18, clip=bounds, label=model)
    reference = 1 if "Cosine" in score_name else 0
    axis.axvline(reference, color="black", linestyle="--", linewidth=1, label="Ideal")
    axis.set_title(score_name, fontsize=11)
    axis.set_xlabel("Similarity" if "Cosine" in score_name else "KL Distance")
    axis.set_ylabel("Density")
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

axes[0].legend(frameon=False)
plt.show()